# ViT-S/16 Room Classification — Part 3: self-labelling (Colab runner)

Semi-supervised **self-training**: warm-start from a supervised model, then add the
**unlabelled** pool (the held-out 1500/class), self-label it in real time with the
network's current knowledge (confidence threshold τ), and train on labelled +
confident pseudo-labels. This notebook is **separate** from `run_colab.ipynb`
(Parts 1–2), which stays runnable on its own.

Run the cells top to bottom on a **T4 GPU** runtime.

## 1. Check the GPU

In [2]:
!nvidia-smi

Tue Jul 14 17:54:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Get the code (clone from GitHub) and install deps
Part 3 is not on `main` yet, so this checks out the **`part3-self-labeling`** branch (where `self_train.py` and the self-labelling modules live). After the PR is merged, you can switch `BRANCH` back to `main`.

In [3]:
import os

REPO_DIR = '/content/room-classification'
BRANCH   = 'part3-self-labeling'   # Part 3 lives on this branch (NOT yet merged to main)

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/SilviuBR24/room-classification.git {REPO_DIR}
# Check out the Part-3 branch and update it (so self_train.py etc. are present).
!cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}

%cd {REPO_DIR}/vit_s16_baseline
!pip install -q -r requirements.txt
print('Code ready on branch:', BRANCH)

Cloning into '/content/room-classification'...
remote: Enumerating objects: 30722, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 30722 (delta 54), reused 77 (delta 28), pack-reused 30612 (from 1)
Receiving objects: 100% (30722/30722), 336.68 MiB | 19.45 MiB/s, done.
Resolving deltas: 100% (57/57), done.
Updating files: 100% (30638/30638), done.
From https://github.com/SilviuBR24/room-classification
 * branch              part3-self-labeling -> FETCH_HEAD
Branch 'part3-self-labeling' set up to track remote branch 'part3-self-labeling' from 'origin'.
Switched to a new branch 'part3-self-labeling'
From https://github.com/SilviuBR24/room-classification
 * branch              part3-self-labeling -> FETCH_HEAD
Already up to date.
/content/room-classification/vit_s16_baseline
Code ready on branch: part3-self-labeling


## 3. Mount Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 4. Unzip the split dataset (train / val / unlabeled / test)
Same `dataset_split.zip` as the baseline notebook. Part 3 uses `unlabeled/` (1500/class)
as the pool to self-label, `train/` (3200) as the real labels, `val/` for selection,
`eval/` as the held-out test.

In [5]:
import os, time, shutil, zipfile

DRIVE_ZIP = '/content/drive/MyDrive/Dissertation_Thesis/dataset_split.zip'
LOCAL_ZIP = '/content/dataset_split.zip'
DATA_ROOT = '/content/dataset_split'

if not os.path.isdir(os.path.join(DATA_ROOT, 'train')):
    if not os.path.exists(LOCAL_ZIP):
        t0 = time.time(); shutil.copy(DRIVE_ZIP, LOCAL_ZIP)
        print(f'Copied zip Drive->local in {time.time()-t0:.0f}s')
    t1 = time.time()
    with zipfile.ZipFile(LOCAL_ZIP) as z:
        for info in z.infolist():
            name = info.filename.replace('\\', '/')   # Windows backslash -> /
            if name.endswith('/'):
                continue
            target = os.path.join('/content', name)
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(info) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)
    print(f'Extracted (backslash-safe) in {time.time()-t1:.0f}s')
else:
    print(f'{DATA_ROOT} already present, skipping.')

for sub in ['train', 'val', 'unlabeled', 'eval']:
    p = os.path.join(DATA_ROOT, sub)
    counts = {c: len(os.listdir(os.path.join(p, c))) for c in sorted(os.listdir(p))}
    print(f'{sub:9s}:', counts)

Copied zip Drive->local in 12s
Extracted (backslash-safe) in 5s
train    : {'bathroom': 3200, 'bedroom': 3200, 'dining_room': 3200, 'entrance_hall': 3200, 'kitchen': 3200, 'living_room': 3200}
val      : {'bathroom': 300, 'bedroom': 300, 'dining_room': 300, 'entrance_hall': 300, 'kitchen': 300, 'living_room': 300}
unlabeled: {'bathroom': 1500, 'bedroom': 1500, 'dining_room': 1500, 'entrance_hall': 1500, 'kitchen': 1500, 'living_room': 1500}
eval     : {'bathroom': 100, 'bedroom': 100, 'dining_room': 100, 'entrance_hall': 100, 'kitchen': 100, 'living_room': 100}


## 5. Build the self-training config and run (automatic, live logs)
Pick the experiment with `EXPERIMENT`:

- **`exp1`** — the spec scenario: 3200 labelled + 1500 unlabelled, **warm-started** from the clean center-loss run. `use_center_loss=True` (spec keeps center loss on).
- **`exp2`** — few-labels ablation: only 500/class labelled, the remaining 2700 + the 1500 join the unlabelled pool (no warm-start; the loop self-bootstraps on the 500). Run it **twice** via `EXP2_VARIANT`: `'semi'` (self-labelling) and `'supervised'` (control, `tau=1.01` → no pseudo-label ever accepted). Distinct `run_name`s so they don't collide; compare semi vs supervised on the test set.

Logs stream **one clean line per (round, epoch)** (`TQDM_DISABLE=1` + `python -u`).

In [ ]:
import os, sys, glob, yaml, subprocess

PROJECT   = '/content/room-classification/vit_s16_baseline'
DATA_ROOT = '/content/dataset_split'
RUNS_DIR  = '/content/drive/MyDrive/Dissertation_Thesis/dissertation_runs'
if os.path.isdir(PROJECT):
    os.chdir(PROJECT)

EXPERIMENT = 'exp1'   # 'exp1' (3200 + 1500, spec)  OR  'exp2' (few-labels 500/class)

cfg = yaml.safe_load(open('config.yaml'))
cfg['data']['train_dir'] = DATA_ROOT + '/train'
cfg['data']['val_dir']   = DATA_ROOT + '/val'
cfg['data']['eval_dir']  = DATA_ROOT + '/eval'
cfg['paths']['output_root']    = RUNS_DIR
cfg['training']['batch_size']  = 64
cfg['training']['num_workers'] = os.cpu_count()
cfg['training']['use_amp']     = True

ssl = dict(unlabeled_dir=DATA_ROOT + '/unlabeled', tau=0.95, rounds=6, epochs_per_round=5,
           use_center_loss=True, center_loss_weight=0.0005, center_loss_lr=0.5,
           labeled_per_class=None, learning_rate=None)   # None -> keep config LR (3e-4)

warmup = ''
if EXPERIMENT == 'exp1':
    cfg['run_name'] = 'vit_s16_3200_selflabel_center'
    ssl['learning_rate'] = 3e-5        # gentle LR: fine-tune the warm-started model
    cks = sorted(glob.glob(RUNS_DIR + '/*center_l00005/checkpoints/best_model.pt'))
    assert cks, 'center l00005 best_model.pt not found on Drive (run the baseline notebook first).'
    warmup = cks[-1]
else:  # exp2 -- few-labels: only 500/class labelled, the rest (2700 + 1500) -> unlabelled pool
    EXP2_VARIANT = 'semi'   # 'semi' = self-labelling  OR  'supervised' = control (no pseudo-labels)
    ssl['labeled_per_class'] = 500
    # no warm-start -> keep the from-scratch LR (learning_rate stays None)
    if EXP2_VARIANT == 'supervised':
        ssl['tau'] = 1.01                          # nothing ever passes -> pure supervised-500 control
        cfg['run_name'] = 'vit_s16_exp2_500_supervised'
    else:
        cfg['run_name'] = 'vit_s16_exp2_500_semi'

cfg['ssl'] = ssl
yaml.safe_dump(cfg, open('config_selftrain.yaml', 'w'), sort_keys=False)
print('EXPERIMENT =', EXPERIMENT, '| run_name =', cfg['run_name'],
      '| tau =', ssl['tau'], '| lr =', ssl['learning_rate'] or cfg['training']['learning_rate'],
      '| warmup =', warmup or '(none)')

CHILD_ENV = {**os.environ, 'TQDM_DISABLE': '1', 'PYTHONUNBUFFERED': '1'}
cmd = ['python', '-u', 'self_train.py', '--config', 'config_selftrain.yaml']
if warmup:
    cmd += ['--warmup', warmup]
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=CHILD_ENV)
for line in p.stdout:
    sys.stdout.write(line.decode('utf-8', 'replace')); sys.stdout.flush()
assert p.wait() == 0, 'self_train.py failed'

## 6. Evaluate the self-trained model on the TEST set
Set `RUN_TAG` to match the experiment you ran. Saves confusion matrix + per-class report
+ embeddings (for later t-SNE).

In [ ]:
import os, sys, glob, subprocess

RUNS_DIR = '/content/drive/MyDrive/Dissertation_Thesis/dissertation_runs'
TEST_DIR = '/content/dataset_split/eval'
RUN_TAG  = 'vit_s16_3200_selflabel_center'   # exp1
# exp2 semi:       'vit_s16_exp2_500_semi'
# exp2 supervised: 'vit_s16_exp2_500_supervised'

runs = sorted(glob.glob(f'{RUNS_DIR}/*{RUN_TAG}/'))
assert runs, f'no run found for {RUN_TAG}'
best = os.path.join(runs[-1], 'checkpoints', 'best_model.pt')
print('Evaluating', RUN_TAG, '->', best)

env = {**os.environ, 'TQDM_DISABLE': '1', 'PYTHONUNBUFFERED': '1'}
p = subprocess.Popen(['python', '-u', 'evaluate.py', '--checkpoint', best,
                      '--data-dir', TEST_DIR, '--save-embeddings'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
for line in p.stdout:
    sys.stdout.write(line.decode('utf-8', 'replace')); sys.stdout.flush()
p.wait()